# **Sesión 3:** Procesamiento de imagenes y extracción de características

## **Librerías**

In [32]:
import cv2
print("OpenCV should be 4.8.0.76 Current version:", cv2.__version__)
from typing import List
import numpy as np
import imageio
import copy
import glob
import matplotlib.pyplot as plt
from typing import Optional
from utils import *

OpenCV should be 4.8.0.76 Current version: 4.8.0


## **Apartado A:** Filtro Gaussiano y Detección de bordes: Sobel y Canny

El objetivo de este apartado es detectar los bordes de las imágenes de la carpeta ``data/partA-B``. Para ello, deberá seguir los siguientes pasos:

1. **Tarea A.1.** Defina el método ``gaussian_blur()`` que aplique un filtro gausiano para obtener imágenes borrosas. Siga todas las indicaciones del enunciado.
2. **Tarea A.2.** Aplique el método ``gaussian_blur()`` a todas las imágenes en ``data/partA-B``.


3. **Tarea A.3.** Defina la función ``sobel_edge_detector()`` que detecte bordes con el método Sobel. Siga todas las indicaciones del enunciado.
4. **Tarea A.4.** Aplique el método ``sobel_edge_detector()`` a todas las imágenes en ``data/partA-B``.


5. **Tarea A.5.** Defina la función ``canny_edge_detector()`` que detecte bordes con el método Canny. Siga todas las indicaciones del enunciado.
6. **Tarea A.6.** Aplique el método ``canny_edge_detector()`` a todas las imágenes en ``data/partA-B``.

In [33]:
# Nos definimos esta función auxiliar que nos facilitará mostrar imágenes más adelante
def show_image(img: np.array, img_name: str = "Image"):
    cv2.imshow(img_name, img)  # Mostramos la imagen, poniendole el nombre correspondiente a la ventana desplegada
    cv2.waitKey(0)  # Ponemos una tecla para eliminar la ventana y pasar a la siguiente imagen o terminar la visualización
    cv2.destroyAllWindows()

In [34]:
# Nos definimos esta función auxiliar que nos facilitará cargar y leer imágenes más adelante
def load_images(filenames: List) -> List:
    return [cv2.imread(filename) for filename in filenames]

### **Tarea A.1:** Defina el método ``gaussian_blur()`` que aplique un filtro gausiano para obtener imágenes borrosas.

In [35]:
# TODO Define the method
def gaussian_blur(img: np.array, sigma: float, filter_shape: Optional[List] = None, verbose: bool = False) -> np.array:
    # TODO If not given, compute the filter shape 

    # Para evitar aplicar ningun filtro gaussiano (útil para comparar imágenes con y sin ruido de apartados posteriores)
    if sigma == 0:
        return img

    # Dependiendo de si el parámetro del tamaño del kernel se pasa como parámetro o no, sacamos las dimensiones del kernel de una o de otra forma
    if filter_shape == None:
        k = max(3, int(6*sigma + 1))  # en caso de no tener el tamaño del kernel, lo sacamos en función de la sigma del kernel
        if k % 2 == 0:
            k += 1  # Nos aseguramos de que sea un valor impar
        kh,kw = k,k

    else:  # En caso de sí disponer del tamaño del kernel, simplemente lo cogemos
        kh, kw = int(filter_shape[0]), int(filter_shape[1])
        if kh // 2 == 1: kh +=1
        if kw // 2 == 1: kw += 1  # Nos aseguramos de que sea un valor impar

    # TODO Create the filter coordinates matrices
    cy,cx = kh // 2, kw // 2
    y, x = np.mgrid[-cy:kh-cy, -cx:kw-cx]  # la función de np.mgrid genera dos arrays 2D con los índices de cada coordenada
    
    # TODO Define the formula that goberns the filter
    formula = (1 / (2*np.pi*sigma**2))*np.exp(-(x**2+y**2)/(2*sigma**2))  # Definimos la fórmula de la Gaussiana pero aún no la hemos normalizado
    
    gaussian_filter = (formula / formula.sum()).astype(np.float32) # Aseguramos que la suma sea 1 (normalizando) para que la convolución no cambie el brillo global
    
    # TODO Process the image
    gb_img = cv2.filter2D(img, -1, gaussian_filter)  # Aplicamos el filtro gaussiano de blurreado a la imagen que se nos había pasado como parámetro
    
    if verbose:
        show_image(img=gb_img, img_name=f"Gaussian Blur: Sigma = {sigma}")
    
    # Devolvemos tanto el filtro gaussiano de blurreado que hemos generado, como la imagen a la que se lo hemos aplicado
    return gaussian_filter, gb_img.astype(np.uint8)

### **Tarea A.2.** Aplique el método ``gaussian_blur()`` a todas las imágenes en ``data``.

In [36]:
imgs_gen_path = '../data/partA-B/*.jpg' # Patrón que cumplen todos los paths de los archivos de imágenes de la carpeta data/partA-B
imgs_path = sorted(glob.glob(imgs_gen_path))  # Cogemos todos los paths que cumplen ese patrón genérico (usando glob) y los ordenamos (usando sorted)
imgs = load_images(imgs_path)  # Le pasamos a load_images la lista de paths o filenames de las imágenes que queremos cargar

imgs_path

['../data/partA-B\\000.jpg',
 '../data/partA-B\\001.jpg',
 '../data/partA-B\\002.jpg',
 '../data/partA-B\\003.jpg',
 '../data/partA-B\\004.jpg',
 '../data/partA-B\\005.jpg']

In [37]:
# # Mostramos las imágenes recién cargadas sin aplicarles nada
for i, img in enumerate(imgs):
    show_image(img = img, img_name = f"Image_{i}")

In [38]:
# TODO Get the gaussian blurred images using a list comprehension
gauss_sigma = 1.4   # Decidimos usar un valor de sigma moderado que consiga aplicar el blur sin llegar a perder demasiada información de la imagen
# Además es el valor que se usa en el ejemplo de las diapositivas del tema 3, por lo que lo usamos como referencia
gb_imgs = [gaussian_blur(img,gauss_sigma,None,False)[1] for img in imgs]  # Aplicamos el filtro a cada imagen y guardamos el resultado

# Queremos blurrear para quitar el ruido y seguir procesando la imagen, ese es el objetivo de este primer paso con el filtro gaussiano

In [39]:
# Mostramos los resultados de las imágenes que hemos blurreado con el filtro Gaussiano
for i, img in enumerate(gb_imgs):
    show_image(img = img, img_name = f"Blurred_Image_{i}")

### **Tarea A.3:** Defina la función ``sobel_edge_detector()`` que detecte bordes con el método Sobel.

In [40]:
# TODO Define the method

def sobel_edge_detector(img: np.array, filter: np.array, gauss_sigma: float, gauss_filter_shape: Optional[List] = None, verbose: bool = False) -> np.array:
    # TODO Transform the img to grayscale
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # TODO Get a blurry img to improve edge detections
    blurred = gaussian_blur(img=gray_img, sigma=gauss_sigma, filter_shape=gauss_filter_shape, verbose=verbose)[1]
    # Accedemos al índice 1 ya que recordemos que la función de gaussian_blur devuelve el filtro usado y la imagen resultado
    
    # Re-scale
    blurred = blurred/255
    # Estandarizamos antes de calcular derivadas y ángulos ya que queremos trabajar siempre con magnitudes comparables y mantener esa consistencia
    
    # TODO Get vertical edges
    v_edges = cv2.filter2D(blurred, -1, filter)  # Usamos el filtro "filter" para obtener los bordes verticales
    
    # TODO Transform the filter to get the orthogonal edges
    filter = filter.T # para obtener el filtro de bordes horizontales, solo tenemos que trasponer el filtro que teníamos para los verticales
    
    # TODO Get horizontal edges
    h_edges = cv2.filter2D(blurred, -1, filter)
    
    # TODO Get edges
    # Calculamos la magintud del gradiente en cada punto, como la suma de los cuadrados del gradiente horizontal y del vertical
    sobel_edges_img = np.hypot(v_edges, h_edges)  
    # Cuanto mayor sea el gradiente en ese punto, es que el cambio de intensidad es mayor y con más probabilidad sea un borde 
    
    # Get edges angle
    theta = np.arctan2(h_edges, v_edges) # Calculamos el ángulo usando la arcotangente de la parte vertical entre la horizontal
    # para posteriores etapas, nos es útil conocer la dirección/orientación en la que va el gradiente, es decir en la que cambia la intensiad    

    # Visualize if needed
    if verbose:
        show_image(img=sobel_edges_img, img_name="Sobel Edges")
    
    # la función de squeeze elimina posibles ejes de longitud 1 que se nos hayan quedado por ahí al hacer operaciones con los arrays
    return np.squeeze(sobel_edges_img), np.squeeze(theta)

### **Tarea A.4.** Aplique el método ``sobel_edge_detector()`` a todas las imágenes en ``data``.

In [41]:
# TODO Define a sigma value
gauss_sigma = 1.4

# TODO Define the Sobel filter
sobel_filter = (1/8)*np.array([[-1, 0, 1],
                         [-2, 0, 2],
                         [-1, 0, 1]], dtype=np.float32)   # Usamos el kernel indicado en el enunciado (es el que se usa para detectar bordes verticales)


# TODO Get the edges detected by Sobel using a list comprehension
sobel_edges_imgs = [sobel_edge_detector(img=img, filter=sobel_filter, gauss_sigma=gauss_sigma, gauss_filter_shape=None, verbose=True)[0]
                    for img in imgs]

In [42]:
# Vemos que las dimensiones de las imágenes con los bordes detectados coinciden con las dimensiones de las imágenes iniciales
print(sobel_edges_imgs[0].shape == imgs[0].shape)
print(sobel_edges_imgs[0].shape)

False
(184, 275)


In [43]:
# Mostramos los resultados de las imágenes a las que hemos aplicado el detector de bordes de Sobel
for i, img in enumerate(sobel_edges_imgs):
    show_image(img = img, img_name = f"Edge_Detection_Sobel_Image_{i}")


"""
TO DO:

los bordes no se ven igual de bien que en la imagen del enunciado del guion de laboratorio, ¿que podemos hacer para mejorarlo?

igual cambiar la sigma (creo que no) o cambiar el tamaño del kernel y definir uno nosotros a mano?

"""

'\nTO DO:\n\nlos bordes no se ven igual de bien que en la imagen del enunciado del guion de laboratorio, ¿que podemos hacer para mejorarlo?\n\nigual cambiar la sigma (creo que no) o cambiar el tamaño del kernel y definir uno nosotros a mano?\n\n'

### **Tarea A.5:** Defina la función ``canny_edge_detector()`` que detecte bordes con el método Canny.

In [44]:
# TODO Define the method
def canny_edge_detector(img: np.array, sobel_filter: np.array, gauss_sigma: float, gauss_filter_shape: Optional[List] = None, verbose: bool = False):
    # TODO Call the method sobel_edge_detector()
    sobel_edges_img, theta = sobel_edge_detector(img=img, filter=sobel_filter, gauss_sigma=gauss_sigma, gauss_filter_shape=gauss_filter_shape, verbose=verbose)
    
    # TODO Use NMS to refine edges
    canny_edges_img = non_max_suppression(img=sobel_edges_img, theta=theta)  
    # La función de non max supresion nos proporciona una imagen que ya tiene los bordes detectados, pero aún más refinados
    
    if verbose:
        show_image(canny_edges_img, img_name="Canny Edges")
        
    return canny_edges_img

### **Tarea A.6.** Aplique el método ``canny_edge_detector()`` a todas las imágenes en ``data``.

In [45]:
# TODO Define Sobel filter
sobel_filter = (1/8)*np.array([[-1, 0, 1],
                         [-2, 0, 2],
                         [-1, 0, 1]], dtype=np.float32)   # Usamos el kernel indicado en el enunciado (es el que se usa para detectar bordes verticales)

# TODO Define a sigma value for Gauss
gauss_sigma = 1.4

# TODO Define a Gauss filter shape
gauss_filter_shape = None

# TODO Get the edges detected by Canny using a list comprehension
canny_edges = [canny_edge_detector(img=img, sobel_filter=sobel_filter, gauss_sigma=gauss_sigma, gauss_filter_shape=gauss_filter_shape, verbose=True)
               for img in imgs]

In [46]:
# Mostramos los resultados de las imágenes a las que hemos aplicado el detector de bordes de Canny
for i, img in enumerate(canny_edges):
    show_image(img = img, img_name = f"Edge_Detection_Canny_Image_{i}")


### **Pregunta A.1:** Añada ruido a las imágenes de la carpeta ``data``. Compare los resultados que obtiene al aplicar su filtro Sobel con y sin filtro Gausiano.

In [47]:
# # TODO Homework

# Nos creamos una función que automáticamente añade ruido gaussiano a las imágenes
def add_gaussian_noise(img, mean=0, sigma=25):
    noisy = img + np.random.normal(mean, sigma, img.shape)
    noisy = np.clip(noisy, 0, 255).astype(np.uint8)  # aseguramos que se quede en un rango que no sea válido
    return noisy

# Probamos a aplicar la función a una de nuestras imágenes
noisy_gauss = add_gaussian_noise(imgs[0], sigma=25)
show_image(noisy_gauss)


In [48]:
# Creamos nuestra lista de imágenes con ruido
noisy_images = [add_gaussian_noise(img) for img in imgs]

# Mostramos los resultados de las imágenes a las que hemos aplicado la función de añadir ruido
for i, img in enumerate(noisy_images):
    show_image(img = img, img_name = f"Noisy_Image_{i}")

In [49]:
# Aplicamos el filtro de sobel a la lista de imágenes ruidosas pero con y sin filtro gaussiano (para blurrear)

# Sigma para cuando sí que aplicamos el filtro gaussiano
sigma_with_gaussian_filter = 1.4

# Sigma para cuando no aplicamos el filtro gaussiano
sigma_without_gaussian_filter = 0

# Filtro de Sobel que vamos a usar
sobel_filter = (1/8)*np.array([[-1, 0, 1],
                         [-2, 0, 2],
                         [-1, 0, 1]], dtype=np.float32)   # Usamos el kernel indicado en el enunciado (es el que se usa para detectar bordes verticales)


In [53]:
# Bordes detectados por Sobel usando el filtro gaussiano
sobel_edges_noisy_imgs_with_gaussian_filter = [sobel_edge_detector(img=img, filter=sobel_filter, 
                                                                   gauss_sigma=sigma_with_gaussian_filter, 
                                                                   gauss_filter_shape=None, verbose=True)[0]
                    for img in noisy_images]


In [54]:
# Bordes detectados por Sobel sin usar el filtro gaussiano
sobel_edges_noisy_imgs_with_gaussian_filter = [sobel_edge_detector(img=img, filter=sobel_filter, 
                                                                   gauss_sigma=sigma_without_gaussian_filter, 
                                                                   gauss_filter_shape=None, verbose=True)[0]
                    for img in noisy_images]

In [52]:
a

NameError: name 'a' is not defined

### **Pregunta A.2:** Utilice la librería ``scikit-image`` y compare el efecto de los filtros Sobel, Canny y Prewitt sobre las imágenes de la carpeta ``data``. ¿Qué diferencias observa entre los filtros? ¿Puede obtener alguna conclusión y/o patrón?

In [ ]:
# TODO Homework

## **Apartado B:** Operadores Morfológicos

Para resolver este partado, deberá seguir los siguientes pasos:

1. **Tarea B.1.** Defina el método ``binarize()`` para binarizar imágenes.
2. **Tarea B.2.** Defina el método ``custom_dilate()``.
3. **Tarea B.3.** Defina el método ``custom_erode()``.
4. **Pregunta B.1** Aplique los métodos ``custom_dilate()`` y ``custom_erode()`` a todas las imágenes de la carpeta ``data``.


### **Tarea B.1.** Defina el método ``binarize()`` para binarizar imágenes.

In [ ]:
# TODO Homework: define the binarization method
def binarize(img: np.array, threshold: int = 127):
    binary_img = None
    return binary_img

### **Tarea B.2.** Defina el método ``custom_dilate()``

In [ ]:
# TODO Homework: define the dilation method
def custom_dilate(img):
    # TODO pad the original image so it can keep dimensions after processing
    padded = np.pad()
    
    # TODO get img shape
    width = None
    height = None
    
    # TODO Create an element with the same dimensions as the padded img
    dilated = np.zeros()
    
    for j in range(height):
        for i in range(width):
            # TODO Add logic to the operation
            pass
            
    # TODO Select the region of interest (ROI). Modify if needed
    dilated = dilated[1:height+1, 1:width+1]
    
    return dilated

### **Tarea B.3.** Defina el método ``custom_erode()``

In [ ]:
# TODO Homework: define the erotion method
def custom_erode(img):
    eroded = None
    
    return eroded

### **Pregunta B.1** Aplique los métodos ``custom_dilate()`` y ``custom_erode()`` a todas las imágenes de la carpeta ``data``.

In [ ]:
# TODO Homework